**Author**: Felipe Matheus  
**Start Date**: 06/05/2026  
**End Date**: 26/08/2026  
**Related Links**:

- Results so far: https://docs.google.com/spreadsheets/d/1OnIWY_bKxgdGU3trECDTL971WfXFPWgbW-dQzgP6KhI/edit?gid=0#gid=0
- Dataset used: https://drive.google.com/drive/u/0/folders/1VsU-taxtwfmDLUu5Z4JmdgXSoxUoB8FI

**Objective**

Build an uncertainty-aware probabilistic predictor of conductivity (IACS) after the **annealing** process, using AutoGluon. The output of every prediction is a Gaussian-like predictive distribution $\mathcal{N}(\hat\mu, \hat\sigma^2)$ where the variance is decomposed into epistemic and aleatoric components.

**Pipeline**

1. Setup and global configuration.
2. Load + preprocess data.
3. Train **Model A** (mean predictor with internal K-fold bagging).
4. Extract OOF predictions per base learner.
5. Recover ensemble weights via NNLS; compute weighted $\hat\mu$ and $\hat\sigma^2_{\text{epist}}$.
6. Build aleatoric targets $\tilde r^2 = \max(r^2_{\text{OOF}} - \hat\sigma^2_{\text{epist}}, 0)$.
7. Train **Model B** on $\log(\tilde r^2 + \epsilon)$.
8. Calibration diagnostics and scalar recalibration.

# 1. Setup

In [ ]:
import os
import sys
import pickle

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from autogluon.tabular import TabularPredictor

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.modeling.Modeling import Modeling
from src.metrics.Evaluation import Evaluation
from src.DataLoader import LoaderHelper
from src.pipelines.AnnealingPipeline import AnnealingDataPipeline

from config.Variables import Variables
from config.bags_compositions import ALL_SAMPLES
from config.elements_coefficients import RESISTIVITY_FACTORS, ENRICHMENT_FACTORS

%load_ext autoreload
%autoreload 2

In [ ]:
# Helper instances reused across the notebook.
modl = Modeling()
evla = Evaluation()
varv = Variables()
load = LoaderHelper()

## 1.1 Global configuration

All knobs that may change between surrogates / processes / experiments live here. Other notebooks for other surrogates only need to edit this cell.

In [ ]:
# ---- Experiment settings ----
ANNEALING_SCHEMA_DATE = "051026"
SN_SCHEMA_DATE = "051026"

ID = "LT-20"
VARIABLE = "iacs"
PROCESS = f"annealing_iacs"
TAG = f"model_{PROCESS}_{ANNEALING_SCHEMA_DATE}_{ID}_measured-diameter"

# ---- Paths ----
FILE_NAME = f"dataset_{PROCESS}.csv"
FILE_NAME_SCHEMA_DATA = f"schema_annealing_essays_{ANNEALING_SCHEMA_DATE}.csv"
FILE_NAME_VALIDATION_DATA = "Experimental Results Annealing V2 - CompiledResults.csv"
FILE_NAME_SN_COMPOSITION = f"serial-number-encoding_{SN_SCHEMA_DATE}.csv"

# All artifacts of THIS run (predictors included) live under the TAG folder,
# so different experiments never overwrite each other's models.
EXPERIMENT_DIR = os.path.join(varv.PATHS.models, PROCESS, TAG)
os.makedirs(EXPERIMENT_DIR, exist_ok=True)

# ---- Dataset schema ----
TARGET = "iacs_final"
FEATURES = ["measured_initial_diameter", "IRI", "purity", "iacs", "temperature", "time"]
WEIGHT_ON_ESSAY_ROWS = 1

# ---- AutoGluon training knobs (Model A) ----
PRESETS_A = "best_quality"     # 'best_quality' for final runs only or medium_quality
NUM_BAG_FOLDS_A = 5               # K-fold bagging produces OOF preds
NUM_BAG_SETS_A = 1
NUM_STACK_LEVELS_A = 0            # no multi-level stacking; keep simple
TIME_LIMIT_A = 1500

# ---- AutoGluon training knobs (Model B) ----
PRESETS_B = "best_quality"
NUM_BAG_FOLDS_B = 5
NUM_STACK_LEVELS_B = 0
TIME_LIMIT_B = 300

# ---- Uncertainty pipeline knobs ----
USE_WEIGHTED_VARIANCE = True      # False = uniform mean/var (sanity check)
VARIANCE_FLOOR_FRAC = 0.01        # floor of sigma2_aleat as fraction of Var(y), in case of predictions as "100% sure" it still adds  small variance
CALIBRATION_ALPHAS = (0.5, 0.8, 0.9, 0.95)
RECALIBRATION_TARGET_ALPHA = 0.9

# ---- AGL x H2O comparison ----
USE_SHARED_FOLDS = False           # same fold_id column on both engines
FOLD_SEED = 42

# ---- Physical constraints (inference-time) ----
Y_MAX = 105.0                     # physical ceiling for %IACS

In [ ]:
ANNEALING_FEATURES = [
    "experiment_id",
    "material_reference",
    "purity",
    "measured_initial_diameter",
    "temperature",
    "time",
    "iacs",
    "tensile_strength",
    "elongation",
    "iacs_final",
    "tensile_strength_final",
    "elongation_final",
]

In [ ]:
pipe = AnnealingDataPipeline(
    df_annealing_schema_raw=pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME_SCHEMA_DATA)),
    annealing_features=ANNEALING_FEATURES,
    df_sn=pd.read_csv(os.path.join(varv.PATHS.data_structured, FILE_NAME_SN_COMPOSITION)),
    all_samples=ALL_SAMPLES,    
    resistivity_factors=RESISTIVITY_FACTORS,
    enrichment_factors=ENRICHMENT_FACTORS
)


In [ ]:
TEST_INPUT_SINGLE = {
    "measured_initial_diameter": 1,
    "purity": 99.6985,
    "IRI": 0.542737,
    "iacs": 94.4,
    "temperature": 450+273,
    "time": 60.0,
    }


MATERIAL_PROPERTIES = {
    "measured_initial_diameter": 1,
    "purity": 99.6985,
    "IRI": 0.542737,
    "iacs": 94.4,
    "tensile_strength": 367.73,
    "elongation": 6.39,
}


# 2. Data

In [ ]:
pipe.df_iri_gbei

In [ ]:
pipe.df_sn

In [ ]:
df, df_val = pipe.get_iacs_df_df_val(features_iacs=FEATURES,
                                     target_iacs=TARGET)

In [ ]:
df

In [ ]:
df_val

# 3. Train Model A — Mean Predictor with Bagging

Model A is an AutoGluon ensemble trained with $K$-fold bagging internally. Bagging is non-optional: without it we cannot extract OOF predictions, which are required for honest residuals and for the epistemic-variance estimate.

In [ ]:
predictor_a = modl.fit_model_a(
    df=df,
    target=TARGET,
    features=FEATURES,
    path=os.path.join(EXPERIMENT_DIR, "model_a"),
    presets=PRESETS_A,
    num_bag_folds=NUM_BAG_FOLDS_A,
    num_bag_sets=NUM_BAG_SETS_A,
    num_stack_levels=NUM_STACK_LEVELS_A,
    time_limit=TIME_LIMIT_A,
    # groups="fold_id" if USE_SHARED_FOLDS else None,
    # sample_weight="weight_col",
)


In [ ]:
# Inspect the leaderboard. Last row is WeightedEnsemble_L2; the rest are
# base learners that feed into it.
predictor_a.leaderboard(silent=True)

## 3.1 H2O AutoML Comparison

Quick H2O AutoML run with the same time budget and cross-validation folds as Model A, to benchmark AutoGluon's OOF RMSE against H2O's leader CV RMSE.

In [ ]:
# aml_h2o = modl.fit_h2o_automl(
#     df= df,
#     target= TARGET,
#     features= FEATURES,
#     weight_on_essay_rows= WEIGHT_ON_ESSAY_ROWS,
#     num_bag_folds= NUM_BAG_FOLDS_A,
#     use_shared_folds= USE_SHARED_FOLDS,
#     time_limit= TIME_LIMIT_A
# )

In [ ]:
# aml_h2o.leaderboard

In [ ]:
# NOTE: the previous cell here compared AutoGluon's `score_val` against
# H2O's xval RMSE. That comparison is incoherent when sample weights are
# active: H2O's xval RMSE is WEIGHTED by weights_column, while AutoGluon's
# score_val is UNWEIGHTED unless weight_evaluation=True. The fair,
# like-for-like comparison (unweighted metrics on OOF predictions of both
# engines, same y_true, same folds when USE_SHARED_FOLDS) is done in
# Section 10 ("OOF metrics: AutoGluon vs H2O").
print("See Section 10 for the coherent OOF-vs-OOF comparison.")


# 4. OOF Predictions and Ensemble Weights

Three things are needed before we can compute the epistemic variance:

1. The OOF predictions of the **final ensemble** (one value per row of the train set).
2. The OOF predictions of **each base learner** (one matrix `M × N`).
3. The **weights** that the WeightedEnsemble assigns to each base learner — recovered via NNLS from the OOF matrix and the ensemble OOF vector. The internal AutoGluon API for weights is not stable across versions; NNLS gives a robust route.

In [ ]:
# OOF mean predictions of the final ensemble.
mu_oof = predictor_a.predict_oof()                  # pd.Series, index == df.index
residuals_oof = np.abs(df[TARGET] - mu_oof)
print(f"mu_oof: shape={mu_oof.shape}  any NaN? {mu_oof.isna().any()}")

## 4.1 fast check on big residuals

In [ ]:
df_verification = pd.concat(
    [df[FEATURES + [TARGET]],
     mu_oof.rename(f"{TARGET}_pred"),
     residuals_oof.rename("residual_abs")], axis=1
)

df_verification.sort_values(by="residual_abs", ascending=False)


## 4.2 Recovering weights 

In [ ]:
# OOF matrix per base learner (excludes WeightedEnsemble itself).
oof_matrix, base_model_names = modl.collect_oof_base_learners(predictor_a)
print(f"oof_matrix shape: {oof_matrix.shape}  (M base learners, N rows)")
print("Base learners:", base_model_names)

In [ ]:
# Recover ensemble weights by solving (oof_matrix.T @ w = mu_oof) with w >= 0.
weights, is_recovery_ok, max_diff = modl.recover_ensemble_weights(
    oof_matrix=oof_matrix,
    mu_oof_ensemble=mu_oof.values,
)

print(f"Recovery max diff: {max_diff:.6f}  ({'OK' if is_recovery_ok else 'FALLBACK'})")
print("\nNon-zero weights:")
for name, w in zip(base_model_names, weights):
    if w > 1e-4:
        print(f"  {name:30s}  {w:.4f}")

# 5. Compute $\hat\mu$ and $\hat\sigma^2_{\text{epist}}$

Following the law of total variance, the epistemic component is the spread of the predictions across base learners, weighted by the ensemble weights. The `USE_WEIGHTED_VARIANCE` flag toggles between weighted (the project default) and uniform statistics — the latter is useful as a robustness check, since it does not rely on the NNLS recovery.

In [ ]:
mu_recomputed, sigma2_epist_oof = modl.compute_mu_and_epistemic_variance(
    preds_matrix=oof_matrix,
    weights=weights,
    use_weights=USE_WEIGHTED_VARIANCE,
)

mu_diffs = np.abs(mu_recomputed - mu_oof)
mu_diffs.describe()
# print("sigma2_epist_oof stats:")
# print(f"  mean   = {sigma2_epist_oof.mean():.4f}")
# print(f"  median = {np.median(sigma2_epist_oof):.4f}")
# print(f"  max    = {sigma2_epist_oof.max():.4f}")

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(sigma2_epist_oof, bins=40)
plt.xlabel("sigma2_epist (OOF)")
plt.ylabel("count")
plt.title("Epistemic variance per training row")
plt.tight_layout()
plt.show()

## Assesment comparing with lin

In [ ]:
mu_recomputed_lin, sigma2_epist_oof_lin = modl.compute_mu_and_epistemic_variance(
    preds_matrix=oof_matrix,
    weights=weights,
    use_weights=False,
)

mu_diffs_lin = np.abs(mu_recomputed_lin - mu_oof)
mu_diffs_lin.describe()
# print("sigma2_epist_oof stats:")
# print(f"  mean   = {sigma2_epist_oof_lin.mean():.4f}")
# print(f"  median = {np.median(sigma2_epist_oof_lin):.4f}")
# print(f"  max    = {sigma2_epist_oof_lin.max():.4f}")

In [ ]:
plt.figure(figsize=(7, 3))
plt.hist(sigma2_epist_oof_lin, bins=40)
plt.xlabel("sigma2_epist (OOF)")
plt.ylabel("count")
plt.title("Epistemic variance per training row")
plt.tight_layout()
plt.show()

## Section 5 Conclusion
When number of lines is at least > 40 

Linear shows way more errors between the recomputed values. Weighted one is corrected.

# 6. Build Aleatoric Targets $\tilde r^2$

Each training row gets a noisy point-estimate of the aleatoric variance:

$$\tilde r_i^2 = \max\!\left((y_i - \hat\mu^{\text{OOF}}(\mathbf{x}_i))^2 - \hat\sigma^2_{\text{epist}}(\mathbf{x}_i),\;0\right).$$

Subtracting the epistemic component avoids double-counting it. Truncation at zero handles rows where the ensemble disagreement exceeds the squared residual.

In [ ]:
r_tilde_sq, residuals_oof_arr, diag = modl.build_aleatoric_targets(
    y_true=df[TARGET].values,
    mu_oof=mu_oof.values,
    sigma2_epist_oof=sigma2_epist_oof,
)

print("=== Aleatoric target diagnostics ===")
for k, v in diag.items():
    print(f"  {k:20s} = {v}")

In [ ]:
r_tilde_sq

# 7. Train Model B — Aleatoric Variance Predictor

Model B is trained on $\log(\tilde r^2 + \epsilon)$ to keep the regression target well-conditioned (variances are non-negative and right-skewed). Inference back-transforms via $\exp$.

In [ ]:
predictor_b = modl.fit_model_b(
    df=df,
    features=FEATURES,
    r_tilde_sq=r_tilde_sq,
    path=os.path.join(EXPERIMENT_DIR, "model_b"),
    presets=PRESETS_B,
    num_bag_folds=NUM_BAG_FOLDS_B,
    num_stack_levels=NUM_STACK_LEVELS_B,
    time_limit=TIME_LIMIT_B,
)


In [ ]:
predictor_b.leaderboard(silent=True)

In [ ]:
df_lado_a_lado = pd.DataFrame({
    "residual_oof": residuals_oof,
    "predict_random_error": predictor_b.predict_oof()  , 
    "epistemic_error": sigma2_epist_oof,
    "iacs": df["iacs"],
    "iacs_final": df[TARGET]
}) 
df_lado_a_lado

# 8. Calibration Diagnostics

Test whether the central α-intervals produced by the predictive Gaussian actually contain the true $y$ on a fraction $\approx \alpha$ of OOF points.

- $\alpha$ here is the **nominal coverage** of an interval (between 0 and 1).
- Empirical coverage **below** nominal ⇒ overconfident (intervals too narrow).
- Empirical coverage **above** nominal ⇒ underconfident (intervals too wide).
- A scalar correction $\hat\sigma \mapsto c \cdot \hat\sigma$ pulls the worst level to nominal; we fit $c$ at $\alpha = 0.9$.

In [ ]:
df[TARGET].var()

Real coverage considering sigmas calculated so far

Build the predictive distribution. Not totally OOF cause model b is not generating OOF preds, we assume it's optimistic just a bit too small to be considered.


In [ ]:
variance_floor = VARIANCE_FLOOR_FRAC * df[TARGET].var()
sigma2_aleat_oof = modl.predict_aleatoric_variance(
    predictor_b, df[FEATURES], variance_floor=variance_floor,
)

sigma2_total_oof = sigma2_epist_oof + sigma2_aleat_oof
sigma_total_oof = np.sqrt(sigma2_total_oof)

y_true = df[TARGET].values
cal_before = modl.calibration_table(
    mu=mu_oof.values, sigma=sigma_total_oof, y_true=y_true,
    alphas=CALIBRATION_ALPHAS,
)
print("=== Calibration BEFORE recalibration ===")
cal_before

Fit a scalar recalibration factor to pull coverage using α=0.9 as nominal reference


In [ ]:
c_opt = modl.fit_recalibration_scalar(
    mu=mu_oof.values,
    sigma=sigma_total_oof,
    y_true=y_true,
    target_alpha=RECALIBRATION_TARGET_ALPHA,
)
print(f"Recalibration scalar c = {c_opt:.4f}")

cal_after = modl.calibration_table(
    mu=mu_oof.values, sigma=c_opt * sigma_total_oof, y_true=y_true,
    alphas=CALIBRATION_ALPHAS,
)
print("\n=== Calibration AFTER recalibration ===")
cal_after

# 8.1 Validation Set Evaluation

Evaluate the freshly trained pair (A + B, with the recalibration $c$) on a
validation DataFrame.

`modl.predict_target` is target-agnostic: it accepts a DataFrame **or a dict**
and any AutoGluon predictor (IACS, UTS, Model B...), inferring the features
from the predictor when not given.

In [ ]:
c_opt
weights

In [ ]:
len(base_model_names)
len(weights)

In [ ]:
# ---- Validation set: df_schema (illustrative; in-sample, see caveat) ----
# df_val = df_schema.copy()

# Full calibrated predictive distribution on the validation set.
preds_val = modl.predict_with_uncertainty(
    X=df_val, # df_schema.copy()
    predictor_a=predictor_a,
    predictor_b=predictor_b,
    weights=weights,
    model_names=base_model_names,
    features=FEATURES,
    variance_floor=variance_floor,
    recalibration_c=c_opt,
    use_weights=USE_WEIGHTED_VARIANCE,
    y_max=Y_MAX,
)

val_metrics = evla.metrics_on_dataframe(
    df=df_val, #df_schema.copy()
    target=TARGET,
    mu=preds_val["mu"].to_numpy(),
    sigma=preds_val["sigma_total"].to_numpy(),
    alphas=CALIBRATION_ALPHAS,
)
print("=== Validation metrics (df_schema) ===")
print({k: round(v, 4) for k, v in val_metrics.items() if k != "coverage"})
print("coverage:", {a: round(c, 3) for a, c in val_metrics["coverage"].items()})

In [ ]:
# ---- Target-agnostic point prediction from a plain dict ----
single = modl.predict_target(
    predictor_a,
    TEST_INPUT_SINGLE,
)
print(f"Predicted {TARGET}: {single[0]:.3f}")

# 9. Inference Example

Use the packaged `predict_with_uncertainty` to apply both models to new rows and return a full predictive distribution. The `recalibration_c` argument scales $\hat\sigma$ to match the calibration above.

In [ ]:
df.head(5)

In [ ]:
# Fit the OOD reference on the training inputs (stored in artifacts later).
ood_ref = modl.fit_ood_reference(df, features=FEATURES)

preds = modl.predict_with_uncertainty(
    X=df.head(5),
    predictor_a=predictor_a,
    predictor_b=predictor_b,
    weights=weights,
    model_names=base_model_names,
    features=FEATURES,
    variance_floor=variance_floor,
    recalibration_c=c_opt,
    use_weights=USE_WEIGHTED_VARIANCE,
    y_max=Y_MAX,          # physical ceiling -> mu_trunc / sigma_trunc / p_above_max
    ood_ref=ood_ref,      # inputs outside training cloud get inflated sigma
)
preds


# 10. Persist Artifacts

Save everything that the downstream stages (controller, propagation) will need.

In [ ]:
# ---- Experiment artifacts (everything under EXPERIMENT_DIR / TAG) ----
artifacts_path = os.path.join(EXPERIMENT_DIR, "artifacts.pkl")

artifacts = {
    "features": FEATURES,
    "target": TARGET,
    "base_model_names": base_model_names,
    "weights": weights,
    "variance_floor": variance_floor,
    "recalibration_c": c_opt,
    "use_weighted_variance": USE_WEIGHTED_VARIANCE,
    "calibration_before": cal_before,
    "calibration_after": cal_after,
    "y_max": Y_MAX,
    "ood_ref": ood_ref,
}
with open(artifacts_path, "wb") as f:
    pickle.dump(artifacts, f)

print(f"Saved to: {artifacts_path}")


In [ ]:
# ---- Save global configuration (1.1) as YAML for this experiment (TAG) ----
global_config = {
    "TAG": TAG,
    "ANNEALING_SCHEMA_DATE": ANNEALING_SCHEMA_DATE,
    "varv.PATHS.data_raw": varv.PATHS.data_raw,
    "varv.PATHS.models": varv.PATHS.models,
    "FILE_NAME": FILE_NAME,
    "FILE_NAME_SCHEMA_DATA": FILE_NAME_SCHEMA_DATA,
    "PROCESS": PROCESS,
    "TARGET": TARGET,
    "FEATURES": FEATURES,
    "WEIGHT_ON_ESSAY_ROWS": WEIGHT_ON_ESSAY_ROWS,
    "PRESETS_A": PRESETS_A,
    "NUM_BAG_FOLDS_A": NUM_BAG_FOLDS_A,
    "NUM_BAG_SETS_A": NUM_BAG_SETS_A,
    "NUM_STACK_LEVELS_A": NUM_STACK_LEVELS_A,
    "TIME_LIMIT_A": TIME_LIMIT_A,
    "PRESETS_B": PRESETS_B,
    "NUM_BAG_FOLDS_B": NUM_BAG_FOLDS_B,
    "NUM_STACK_LEVELS_B": NUM_STACK_LEVELS_B,
    "TIME_LIMIT_B": TIME_LIMIT_B,
    "USE_WEIGHTED_VARIANCE": USE_WEIGHTED_VARIANCE,
    "VARIANCE_FLOOR_FRAC": VARIANCE_FLOOR_FRAC,
    "CALIBRATION_ALPHAS": list(CALIBRATION_ALPHAS),
    "RECALIBRATION_TARGET_ALPHA": RECALIBRATION_TARGET_ALPHA,
    "USE_SHARED_FOLDS": USE_SHARED_FOLDS,
    "FOLD_SEED": FOLD_SEED,
    "Y_MAX": Y_MAX,
}

config_path = os.path.join(EXPERIMENT_DIR, "global_config.txt")
with open(config_path, "w") as f:
    yaml.dump(global_config, f, default_flow_style=False, sort_keys=False)

print(f"Saved to: {config_path}")


In [ ]:
# ---- OOF metrics: AutoGluon (Model A leader) vs H2O AutoML leader ----
# Both vectors are out-of-fold predictions; metrics are UNWEIGHTED on both
# sides, and with USE_SHARED_FOLDS=True the folds are identical -> coherent.
# h2o_oof_preds = (
#     aml_h2o.leader.cross_validation_holdout_predictions()
#     .as_data_frame()
#     .iloc[:, 0]
#     .to_numpy()
# )

metrics_agl = evla.one_step_metrics(y_true=y_true, mu=mu_oof.values)
# metrics_h2o = evla.one_step_metrics(y_true=y_true, mu=h2o_oof_preds)

metrics_df = pd.DataFrame(
    [
        {"model": "AutoGluon (Model A, leader)", **metrics_agl},
        # {"model": "H2O AutoML (leader)", **metrics_h2o},
    ]
)[["model", "rmse", "mae", "mape", "r2"]]

print(metrics_df)

# ---- Leaderboards from both engines ----
leaderboard_agl = predictor_a.leaderboard(silent=True)
# leaderboard_h2o = aml_h2o.leaderboard.as_data_frame()

# ---- Save everything for this experiment (TAG) into separate CSV files ----
metrics_df.to_csv(os.path.join(EXPERIMENT_DIR, "metrics_comparison.csv"), index=False)
leaderboard_agl.to_csv(os.path.join(EXPERIMENT_DIR, "leaderboard_autogluon.csv"), index=False)
# leaderboard_h2o.to_csv(os.path.join(EXPERIMENT_DIR, "leaderboard_h2o.csv"), index=False)

print(f"Saved to: {EXPERIMENT_DIR}")


# 11. Load Artifacts & Predict (deployment simulation)

This section simulates a fresh process: it loads ONLY what was persisted in
Section 10 (`artifacts.pkl`) plus the two AutoGluon predictors from disk —
no variable from the training session above is reused. This is exactly what a
deployment script (or the MBC controller) would do. The same
`predict_with_uncertainty` call as Section 9 runs, but every ingredient
(`weights`, `recalibration_c`, `ood_ref`, `y_max`, ...) comes from the
reloaded artifacts, proving the persisted bundle is self-contained.

In [ ]:

# ---- 1) Load the persisted artifact bundle (Section 10) ----
loaded_artifacts = load.load_pickle(EXPERIMENT_DIR, "artifacts.pkl")

# ---- 2) Reload both AutoGluon predictors from their TAG subfolders ----
predictor_a_loaded = TabularPredictor.load(
    os.path.join(EXPERIMENT_DIR, "model_a")
)
predictor_b_loaded = TabularPredictor.load(
    os.path.join(EXPERIMENT_DIR, "model_b")
)

print("Loaded artifacts keys:", list(loaded_artifacts.keys()))
print("Target:", loaded_artifacts["target"])
print("Features:", loaded_artifacts["features"])

In [ ]:
# ---- 3) Predict using ONLY the reloaded objects ----
# Build a small query frame (here: first rows of df, but in deployment this
# would be a brand-new candidate route from the controller).
X_query = df.head(5)

preds_loaded = modl.predict_with_uncertainty(
    X=X_query,
    predictor_a=predictor_a_loaded,
    predictor_b=predictor_b_loaded,
    weights=loaded_artifacts["weights"],
    model_names=loaded_artifacts["base_model_names"],
    features=loaded_artifacts["features"],
    variance_floor=loaded_artifacts["variance_floor"],
    recalibration_c=loaded_artifacts["recalibration_c"],
    use_weights=loaded_artifacts["use_weighted_variance"],
    y_max=loaded_artifacts["y_max"],
    ood_ref=loaded_artifacts["ood_ref"],
)
preds_loaded

In [ ]:
# ---- 4) Target-agnostic quick prediction from a plain dict ----
# Works for any predictor/target via modl.predict_target (features inferred
# from the predictor when not passed explicitly).
single_loaded = modl.predict_target(
    predictor_a_loaded,
    TEST_INPUT_SINGLE,
    features=loaded_artifacts["features"],
)
print(f"Predicted {loaded_artifacts['target']}: {single_loaded[0]:.3f}")

In [ ]:
# ---- 5) Sanity check: reloaded predictions match the in-session ones ----
# (Section 9 computed `preds` on df.head(5) with the same ood_ref/c_opt.)
import numpy as np
if "preds" in dir():
    max_mu_diff = float(np.abs(preds["mu"].to_numpy()
                               - preds_loaded["mu"].to_numpy()).max())
    max_sigma_diff = float(np.abs(preds["sigma_total"].to_numpy()
                                  - preds_loaded["sigma_total"].to_numpy()).max())
    print(f"max |mu diff|    = {max_mu_diff:.2e}")
    print(f"max |sigma diff| = {max_sigma_diff:.2e}")
    print("OK: artifacts reproduce the in-session pipeline."
          if max(max_mu_diff, max_sigma_diff) < 1e-8
          else "WARNING: mismatch - check that Section 9 used the same ood_ref.")
else:
    print("Run Section 9 first to enable the sanity check.")

# 12 Adhoc running

In [ ]:
IACS_TAG_DIR = EXPERIMENT_DIR
UTS_TAG_DIR = os.path.join(
    varv.PATHS.models, "annealing_uts", "experiments", "annealing-uts-v1-best_quality"
)
ELONGATION_TAG_DIR = os.path.join(
    varv.PATHS.models, "annealing_elongation", "experiments", "annealing-schema230726"
)

In [ ]:
single_loaded = modl.predict_target(
    predictor_a_loaded,
    TEST_INPUT_SINGLE,
    features=loaded_artifacts["features"],
)
print(f"Predicted {loaded_artifacts['target']}: {single_loaded[0]:.3f}")

In [ ]:
from src.modeling.MBCInference import MBCInference

mbc = MBCInference(modl, load)

# ---- Grid (inclusive of upper bound, like the deterministic build_param_grid) ----
OPERATIONAL_LIMITS = {
    "temperature": [200+273, 600 + 273],  # Kelvin
    "time": [30, 120],  # minutes
}
STEP = {"temperature": 10, "time": 5}

In [ ]:
down_bundles = mbc.load_surrogates_best(
    {
        "iacs": IACS_TAG_DIR,
        "uts": UTS_TAG_DIR,
        "elongation": ELONGATION_TAG_DIR
    },
     metric="rmse",
)

In [ ]:
param_grid = mbc.build_param_grid(OPERATIONAL_LIMITS, STEP)
df_grid = mbc.build_state_grid_df(MATERIAL_PROPERTIES, param_grid)

print(f"Grid: {len(param_grid['temperature'])} temperatures x "
      f"{len(param_grid['time'])} times = {len(df_grid)} cells")
df_grid.head()

In [ ]:
df_pred = mbc.predict_all(down_bundles, df_grid, ood_gamma=1.0)


In [ ]:
df_pred

In [ ]:
df_pred.to_csv(os.path.join(varv.PATHS.data_enriched, f"annealing_predictions_all_{ANNEALING_SCHEMA_DATE}.csv"))

In [ ]:
df_pred.sort_values(by="iacs_mu", ascending=False).head(15)

In [ ]:
df_pred[df_pred["time"] == 60].sort_values(by="iacs_mu", ascending=False)

In [ ]:
df_verification.shape

In [ ]:
df_analysis = df_verification.copy()
df_analysis["iacs_percentual"] = df_analysis["iacs_final"] / df_analysis["iacs"]
df_analysis["iacs_diff"] = df_analysis["iacs_final"] - df_analysis["iacs"]

df_analysis["iacs_percentual_expected"] = df_analysis["iacs_final_pred"] / df_analysis["iacs"]
df_analysis["iacs_diff_expected"] =  df_analysis["iacs_final_pred"] - df_analysis["iacs"]

In [ ]:
df_analysis.columns

In [ ]:
df_analysis

In [ ]:
break

In [ ]:
import importlib


# 1) Colunas derivadas (residual assinado, log_time, delta_rho de Matthiessen)
df_a = add_time_analysis_columns(df_analysis)

# 2) Estatística: Spearman + slope em log(time), POR temperatura
stats = time_trend_stats(df_a)
display(stats.round(4))

# 3) Observed vs expected — o surrogate reproduz a cinética?
fig1 = plot_observed_vs_expected(df_a, metric="iacs_diff")
fig1b = plot_observed_vs_expected(df_a, metric="iacs_percentual")

# 4) O diagnóstico principal: resíduo assinado vs tempo
fig2 = plot_residual_vs_time(df_a)

# 5) Trajetórias por condição (within-condition, sem mistura entre fios)
fig3 = plot_trajectories(df_a, metric="iacs_final")
fig3b = plot_trajectories(df_a, metric="iacs_diff")

# 6) Qual resposta é mais linear em log(time)?
fig4 = plot_metric_comparison(df_a)

In [ ]:
df_analysis["temperature_C"] = df_analysis["temperature"] - 273

In [ ]:

# x = time, observado vs esperado, média
fig, tbl = plot_grouped_response(
    df_analysis, x="time", y=["iacs_diff", "iacs_diff_expected"], agg="mean",
)
display(tbl.round(4))

# x = temperature, mesma dupla
fig, tbl = plot_grouped_response(
    df_analysis, x="temperature_C", y=["iacs_diff", "iacs_diff_expected"], agg="median",
)
display(tbl.round(4))

# versão percentual, mediana (mais robusta a outlier com n pequeno)
fig, tbl = plot_grouped_response(
    df_analysis, x="time",
    y=["iacs_percentual", "iacs_percentual_expected"], agg="median",
)
display(tbl.round(4))

In [ ]:
"""Diagnostics for the time effect on annealing IACS.

Answers two distinct questions that are easy to conflate:

1. **Physics** — does annealing time move ``iacs_final``, and with what
   functional shape (linear? saturating in ``log(time)``?), once
   temperature is held fixed?
2. **Surrogate quality** — does the model *capture* that time effect? A
   signed residual that trends with time means the surrogate is missing
   time structure, regardless of how good its aggregate RMSE looks.

Temperature is a confounder for time in a crossed grid, so every plot
here is stratified by temperature. Aggregate ``time`` correlations
computed across temperatures are reported only as a deliberately
labelled "pooled (confounded)" row.

Matthiessen note
----------------
Scattering contributions add in resistivity, and IACS is inversely
proportional to resistivity. ``delta_rho = 1/iacs_final - 1/iacs`` is
therefore the physically additive counterpart of ``iacs_diff``; if the
time trend is cleaner in ``delta_rho`` than in ``iacs_diff``, that is
evidence the surrogate should see a resistivity-like feature rather
than a raw IACS delta.

Typical use
-----------
>>> df_a = add_time_analysis_columns(df_analysis)
>>> stats = time_trend_stats(df_a)
>>> fig1 = plot_observed_vs_expected(df_a)
>>> fig2 = plot_residual_vs_time(df_a)
"""

from __future__ import annotations

import logging
from typing import Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

logger = logging.getLogger(__name__)

# Columns that are outputs/derived, never part of a "condition" identity.
_NON_CONDITION = {
    "time", "log_time", "iacs_final", "iacs_final_pred", "residual_abs",
    "residual_signed", "residual_rel", "iacs_percentual", "iacs_diff",
    "iacs_percentual_expected", "iacs_diff_expected", "delta_rho",
    "delta_rho_expected", "fold", "weight_col",
}


def add_time_analysis_columns(
    df: pd.DataFrame,
    target: str = "iacs_final",
    pred: str = "iacs_final_pred",
    initial: str = "iacs",
    time_col: str = "time",
) -> pd.DataFrame:
    """Add signed residual, log-time and Matthiessen resistivity deltas.

    Parameters
    ----------
    df : pd.DataFrame
        Frame already carrying ``iacs_diff`` / ``iacs_percentual`` and
        their ``_expected`` counterparts.
    target, pred, initial : str
        Observed target, model prediction, and pre-annealing IACS columns.
    time_col : str
        Annealing time column.

    Returns
    -------
    pd.DataFrame
        Copy of ``df`` with ``residual_signed``, ``residual_rel``,
        ``log_time``, ``delta_rho`` and ``delta_rho_expected``.

    Notes
    -----
    ``residual_signed`` (not ``residual_abs``) is the diagnostic column:
    absolute residuals destroy the sign of the bias, which is exactly
    the information needed to detect a missing time term.
    """
    out = df.copy()
    out["residual_signed"] = out[target] - out[pred]
    out["residual_rel"] = out["residual_signed"] / out[target]
    out["log_time"] = np.log(out[time_col].astype(float))

    # Matthiessen: scattering adds in resistivity, IACS ~ 1 / resistivity.
    out["delta_rho"] = 1.0 / out[target] - 1.0 / out[initial]
    out["delta_rho_expected"] = 1.0 / out[pred] - 1.0 / out[initial]

    logger.info(
        "Added time-analysis columns; n=%d, time range=[%s, %s], "
        "unique times=%d",
        len(out), out[time_col].min(), out[time_col].max(),
        out[time_col].nunique(),
    )
    return out


def infer_condition_columns(
    df: pd.DataFrame, extra_exclude: Sequence[str] = ()
) -> list[str]:
    """Infer the columns that identify one annealing condition.

    A "condition" is everything that is held fixed while time is swept:
    geometry, purity, IRI, initial IACS and temperature. Detected
    structurally (all columns minus known outputs/derived) so no target
    or feature name is hardcoded.

    Parameters
    ----------
    df : pd.DataFrame
        Analysis frame.
    extra_exclude : sequence of str
        Additional columns to treat as non-identity.

    Returns
    -------
    list[str]
        Condition-identifying columns, in frame order.
    """
    exclude = _NON_CONDITION | set(extra_exclude)
    cond = [c for c in df.columns if c not in exclude]
    logger.info("Inferred condition columns: %s", cond)
    return cond


def time_trend_stats(
    df: pd.DataFrame,
    metrics: Sequence[str] = (
        "iacs_final", "iacs_diff", "iacs_percentual", "delta_rho",
        "residual_signed",
    ),
    time_col: str = "time",
    group_col: str = "temperature",
) -> pd.DataFrame:
    """Per-temperature Spearman correlation and log-time OLS slope.

    Parameters
    ----------
    df : pd.DataFrame
        Output of :func:`add_time_analysis_columns`.
    metrics : sequence of str
        Columns to test against time.
    time_col, group_col : str
        Time column and stratification column.

    Returns
    -------
    pd.DataFrame
        One row per (group, metric) with ``n``, ``spearman``, ``p_value``
        and ``slope_log_time``. A final ``pooled (confounded)`` block
        repeats the test ignoring temperature, for contrast only.

    Notes
    -----
    With small per-temperature ``n``, treat ``p_value`` as a rough
    ordering device, not as a decision threshold. The sign and the
    consistency of the sign *across* temperatures carry more evidence
    than any single p-value.
    """
    from scipy import stats as sp_stats

    rows: list[dict] = []

    def _one(sub: pd.DataFrame, label: object, metric: str) -> dict:
        s = sub[[time_col, metric, "log_time"]].dropna()
        n = len(s)
        rho = p = slope = np.nan
        if n >= 3 and s[time_col].nunique() >= 2:
            rho, p = sp_stats.spearmanr(s[time_col], s[metric])
            slope = float(np.polyfit(s["log_time"], s[metric], 1)[0])
        return {
            group_col: label, "metric": metric, "n": n,
            "spearman": rho, "p_value": p, "slope_log_time": slope,
        }

    for label, sub in df.groupby(group_col, sort=True):
        for metric in metrics:
            if metric in df.columns:
                rows.append(_one(sub, label, metric))

    for metric in metrics:
        if metric in df.columns:
            rows.append(_one(df, "pooled (confounded)", metric))

    res = pd.DataFrame(rows)
    logger.info("Computed time-trend stats over %d rows.", len(res))
    return res


def plot_observed_vs_expected(
    df: pd.DataFrame,
    metric: str = "iacs_diff",
    expected_metric: str | None = None,
    time_col: str = "time",
    group_col: str = "temperature",
    figsize_per_panel: tuple[float, float] = (4.2, 3.4),
) -> plt.Figure:
    """Observed vs model-expected time response, one panel per temperature.

    Parameters
    ----------
    df : pd.DataFrame
        Output of :func:`add_time_analysis_columns`.
    metric : str
        Observed metric (e.g. ``iacs_diff`` or ``iacs_percentual``).
    expected_metric : str or None
        Model counterpart; defaults to ``f"{metric}_expected"``.
    time_col, group_col : str
        Time and stratification columns.
    figsize_per_panel : tuple of float
        Size of each subplot.

    Returns
    -------
    matplotlib.figure.Figure

    Notes
    -----
    Read the *gap* between the two curves, not their levels. Curves that
    track each other mean the surrogate reproduces the kinetics; a gap
    that widens with time means it does not.
    """
    exp_col = expected_metric or f"{metric}_expected"
    groups = sorted(df[group_col].dropna().unique())
    ncols = min(3, len(groups))
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, squeeze=False, sharex=True,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows),
    )

    for ax, label in zip(axes.ravel(), groups):
        sub = df[df[group_col] == label]
        obs = sub.groupby(time_col)[metric].agg(["mean", "std", "count"])
        ax.errorbar(
            obs.index, obs["mean"], yerr=obs["std"].fillna(0.0),
            marker="o", capsize=3, label="observed", color="#1f77b4",
        )
        if exp_col in sub.columns:
            exp = sub.groupby(time_col)[exp_col].mean()
            ax.plot(
                exp.index, exp.values, marker="s", linestyle="--",
                label="expected (model)", color="#d62728",
            )
        ax.set_title(f"{group_col} = {label}  (n={len(sub)})", fontsize=10)
        ax.set_xlabel(time_col)
        ax.set_ylabel(metric)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

    for ax in axes.ravel()[len(groups):]:
        ax.set_visible(False)

    fig.suptitle(f"Time response: {metric} observed vs expected", y=1.0)
    fig.tight_layout()
    return fig


def plot_residual_vs_time(
    df: pd.DataFrame,
    residual_col: str = "residual_signed",
    time_col: str = "time",
    group_col: str = "temperature",
    figsize: tuple[float, float] = (11.0, 4.2),
) -> plt.Figure:
    """Signed residual against time — the key surrogate diagnostic.

    Left panel: scatter coloured by temperature with a per-temperature
    mean trace. Right panel: residual against ``log_time``, which
    reveals saturating kinetics that look like noise on a linear axis.

    Parameters
    ----------
    df : pd.DataFrame
        Output of :func:`add_time_analysis_columns`.
    residual_col, time_col, group_col : str
        Column names.
    figsize : tuple of float
        Figure size.

    Returns
    -------
    matplotlib.figure.Figure

    Notes
    -----
    A cloud centred on zero with no tilt at any temperature is the
    "model captured the kinetics" outcome. Any systematic tilt, or a
    tilt whose sign flips between temperatures, indicates a missing
    time--temperature interaction term.
    """
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    groups = sorted(df[group_col].dropna().unique())
    cmap = plt.get_cmap("viridis", max(len(groups), 2))

    for xcol, ax, xlabel in (
        (time_col, axes[0], time_col),
        ("log_time", axes[1], "log(time)"),
    ):
        for i, label in enumerate(groups):
            sub = df[df[group_col] == label]
            ax.scatter(
                sub[xcol], sub[residual_col], s=26, alpha=0.65,
                color=cmap(i), label=f"{group_col}={label}",
            )
            trace = sub.groupby(xcol)[residual_col].mean().sort_index()
            ax.plot(trace.index, trace.values, color=cmap(i), alpha=0.9)
        ax.axhline(0.0, color="black", linewidth=1.2, linestyle="--")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(residual_col)
        ax.grid(alpha=0.3)

    axes[0].legend(fontsize=8)
    fig.suptitle("Signed residual vs time (flat around 0 = time captured)")
    fig.tight_layout()
    return fig


def plot_trajectories(
    df: pd.DataFrame,
    metric: str = "iacs_final",
    time_col: str = "time",
    group_col: str = "temperature",
    condition_cols: Sequence[str] | None = None,
    min_points: int = 2,
    figsize_per_panel: tuple[float, float] = (4.2, 3.4),
) -> plt.Figure:
    """One line per annealing condition, showing its own time sweep.

    Parameters
    ----------
    df : pd.DataFrame
        Output of :func:`add_time_analysis_columns`.
    metric : str
        Column to trace over time.
    time_col, group_col : str
        Time and panel-stratification columns.
    condition_cols : sequence of str or None
        Identity of one condition; inferred via
        :func:`infer_condition_columns` when omitted.
    min_points : int
        Skip conditions with fewer than this many time points.
    figsize_per_panel : tuple of float
        Size of each subplot.

    Returns
    -------
    matplotlib.figure.Figure

    Notes
    -----
    This is the only view where within-condition change is not mixed
    with between-condition variation, so a trend visible here is far
    stronger evidence than one visible in a pooled scatter.
    """
    cond = list(condition_cols) if condition_cols else infer_condition_columns(df)
    cond_no_group = [c for c in cond if c != group_col]

    groups = sorted(df[group_col].dropna().unique())
    ncols = min(3, len(groups))
    nrows = int(np.ceil(len(groups) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, squeeze=False,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows),
    )

    n_traced = 0
    for ax, label in zip(axes.ravel(), groups):
        sub = df[df[group_col] == label]
        key = cond_no_group if cond_no_group else None
        iterator = sub.groupby(key, sort=False) if key else [(None, sub)]
        for _, traj in iterator:
            traj = traj.sort_values(time_col)
            if len(traj) < min_points:
                continue
            ax.plot(
                traj[time_col], traj[metric], marker="o", alpha=0.7,
                linewidth=1.2,
            )
            n_traced += 1
        ax.set_title(f"{group_col} = {label}", fontsize=10)
        ax.set_xlabel(time_col)
        ax.set_ylabel(metric)
        ax.grid(alpha=0.3)

    for ax in axes.ravel()[len(groups):]:
        ax.set_visible(False)

    logger.info("Traced %d conditions with >= %d time points.", n_traced, min_points)
    fig.suptitle(f"Per-condition time trajectories: {metric}", y=1.0)
    fig.tight_layout()
    return fig


def plot_metric_comparison(
    df: pd.DataFrame,
    metrics: Sequence[str] = ("iacs_diff", "iacs_percentual", "delta_rho"),
    time_col: str = "time",
    group_col: str = "temperature",
    figsize_per_panel: tuple[float, float] = (4.2, 3.4),
) -> plt.Figure:
    """Compare candidate response variables against ``log(time)``.

    Whichever metric is closest to a straight line in ``log(time)``, with
    consistent slope across temperatures, is the best-behaved response
    for the surrogate to learn.

    Parameters
    ----------
    df : pd.DataFrame
        Output of :func:`add_time_analysis_columns`.
    metrics : sequence of str
        Candidate response columns.
    time_col, group_col : str
        Column names.
    figsize_per_panel : tuple of float
        Size of each subplot.

    Returns
    -------
    matplotlib.figure.Figure
    """
    metrics = [m for m in metrics if m in df.columns]
    groups = sorted(df[group_col].dropna().unique())
    cmap = plt.get_cmap("viridis", max(len(groups), 2))
    fig, axes = plt.subplots(
        1, len(metrics), squeeze=False,
        figsize=(figsize_per_panel[0] * len(metrics), figsize_per_panel[1]),
    )

    for ax, metric in zip(axes.ravel(), metrics):
        for i, label in enumerate(groups):
            sub = df[df[group_col] == label]
            trace = sub.groupby("log_time")[metric].mean().sort_index()
            ax.plot(
                trace.index, trace.values, marker="o", color=cmap(i),
                label=f"{group_col}={label}",
            )
        ax.set_xlabel("log(time)")
        ax.set_ylabel(metric)
        ax.grid(alpha=0.3)
    axes.ravel()[0].legend(fontsize=8)
    fig.suptitle("Which response is most linear in log(time)?")
    fig.tight_layout()
    return fig


def plot_grouped_response(
    df: pd.DataFrame,
    x: str = "time",
    y: str | Sequence[str] = "iacs_diff",
    agg: str = "mean",
    spread: str | None = "auto",
    annotate_counts: bool = True,
    ax: plt.Axes | None = None,
    figsize: tuple[float, float] = (7.5, 4.8),
    colors: Sequence[str] = ("#1f77b4", "#d62728"),
) -> tuple[plt.Figure, pd.DataFrame]:
    """Aggregate one or two same-dimension columns over ``x`` and plot them.

    The chosen ``x`` (typically ``time`` or ``temperature``) becomes the
    horizontal axis; every row sharing an ``x`` value is collapsed with
    ``agg`` into a single point per curve. Intended for comparing an
    observed column against its model counterpart (e.g. ``iacs_diff``
    vs ``iacs_diff_expected``), which share units and therefore share
    one vertical axis.

    Parameters
    ----------
    df : pd.DataFrame
        Analysis frame.
    x : str
        Grouping column plotted on the horizontal axis.
    y : str or sequence of str
        One or two columns to aggregate. Must share physical units.
    agg : {'mean', 'median'}
        Central-tendency statistic applied within each ``x`` group.
    spread : {'auto', 'std', 'sem', 'iqr', None}
        Shaded band around each curve. ``'auto'`` picks ``'std'`` for
        ``mean`` and ``'iqr'`` for ``median``. ``None`` disables it.
    annotate_counts : bool
        Print the group size under each x tick.
    ax : matplotlib.axes.Axes or None
        Draw into an existing axis when provided.
    figsize : tuple of float
        Figure size when ``ax`` is None.
    colors : sequence of str
        Line colours, one per y column.

    Returns
    -------
    fig : matplotlib.figure.Figure
    table : pd.DataFrame
        Indexed by ``x``, with ``<col>_<agg>``, the spread column and
        ``n`` per group.

    Notes
    -----
    Collapsing over every other column marginalises them. If the grid is
    unbalanced -- different temperatures represented at different times,
    or more replicates at some settings -- the marginal curve mixes those
    compositions and part of the apparent ``x`` trend is composition,
    not physics. The ``n`` annotation exists to make that visible; if the
    counts are uneven, cross-check against the per-temperature panels
    before drawing conclusions.
    """
    if agg not in ("mean", "median"):
        raise ValueError(f"agg must be 'mean' or 'median', got {agg!r}.")
    y_cols = [y] if isinstance(y, str) else list(y)
    if not 1 <= len(y_cols) <= 2:
        raise ValueError(f"Pass one or two y columns, got {len(y_cols)}.")
    missing = [c for c in [x, *y_cols] if c not in df.columns]
    if missing:
        raise KeyError(f"Columns {missing} not in df (have {list(df.columns)}).")

    if spread == "auto":
        spread = "std" if agg == "mean" else "iqr"

    grouped = df.groupby(x, sort=True)
    table = pd.DataFrame(index=grouped.size().index)
    table["n"] = grouped.size()

    fig = ax.get_figure() if ax is not None else plt.figure(figsize=figsize)
    ax = ax if ax is not None else fig.add_subplot(111)

    for i, col in enumerate(y_cols):
        centre = grouped[col].agg(agg)
        table[f"{col}_{agg}"] = centre
        colour = colors[i % len(colors)]
        style = "-" if i == 0 else "--"
        ax.plot(
            centre.index, centre.to_numpy(), marker="o" if i == 0 else "s",
            linestyle=style, color=colour, linewidth=1.8, label=f"{col} ({agg})",
        )

        if spread == "std":
            band = grouped[col].std().fillna(0.0)
            lo, hi = centre - band, centre + band
            table[f"{col}_std"] = band
        elif spread == "sem":
            band = (grouped[col].std() / np.sqrt(grouped[col].count())).fillna(0.0)
            lo, hi = centre - band, centre + band
            table[f"{col}_sem"] = band
        elif spread == "iqr":
            lo = grouped[col].quantile(0.25)
            hi = grouped[col].quantile(0.75)
            table[f"{col}_q25"] = lo
            table[f"{col}_q75"] = hi
        else:
            lo = hi = None

        if lo is not None:
            ax.fill_between(
                centre.index, lo.to_numpy(), hi.to_numpy(),
                color=colour, alpha=0.15, linewidth=0,
            )

    if annotate_counts:
        ymin = ax.get_ylim()[0]
        for xv, n in table["n"].items():
            ax.annotate(
                f"n={n}", xy=(xv, ymin), xytext=(0, -18),
                textcoords="offset points", ha="center", fontsize=7,
                color="gray", annotation_clip=False,
            )

    ax.set_xlabel(x)
    ax.set_ylabel(" / ".join(y_cols))
    ax.set_title(f"{agg} of {' vs '.join(y_cols)} grouped by {x}")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=9)
    fig.tight_layout()

    logger.info(
        "Grouped %d rows by '%s' into %d points (agg=%s, spread=%s).",
        len(df), x, len(table), agg, spread,
    )
    return fig, table

In [ ]:
df_analysis